# 00 — Setup and environment check

Run this once after cloning the repo. It verifies dependencies, checks that the
repo modules import, confirms the Census county shapefile is in place, and
prints a status summary. If every check passes, move on to
`01_build_registry.ipynb`.

## Step 1 — Locate the repo and check imports

In [ ]:
import sys
from pathlib import Path

# This notebook lives in notebooks/; walk up to the repo root.
root = Path.cwd()
while not (root / "config.py").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))
print("repo root:", root)

import config
from base import build_session, decode_polyline, OutageRecord
from kubra import KubraAdapter
from arcgis import ArcGISAdapter
from countyjson import CountyJsonAdapter
from discover import classify_outage_map
from collector import collect_once, load_registry
from county_aggregate import aggregate_to_county_hour
print("all repo modules imported OK")

## Step 2 — Check third-party dependencies

In [ ]:
required = {
    "requests":  "collector + adapters",
    "pandas":    "snapshots + aggregation",
    "pyarrow":   "parquet read/write",
    "geopandas": "county spatial join (notebook 02)",
    "shapely":   "geometry (pulled in by geopandas)",
    "matplotlib":"coverage maps (notebook 02)",
}
missing = []
for pkg, why in required.items():
    try:
        __import__(pkg)
        print(f"  ok    {pkg:<12} - {why}")
    except ImportError:
        missing.append(pkg)
        print(f"  MISS  {pkg:<12} - {why}")
if missing:
    print(f"\nInstall the missing packages:\n  pip install {' '.join(missing)}")
else:
    print("\nall dependencies present")

## Step 3 — Check config and the county shapefile

In [ ]:
print("CONTACT_EMAIL    :", config.CONTACT_EMAIL)
if "your_netid" in config.CONTACT_EMAIL:
    print("  -> edit config.py: set a real contact email before collecting.")
print("OUTAGE_THRESHOLD :", config.OUTAGE_THRESHOLD, "customers")
print("TIME_BLOCK       :", config.TIME_BLOCK)
print("snapshot dir     :", config.SNAPSHOT_DIR)

shp = config.COUNTY_SHAPEFILE
if shp.exists():
    print("county shapefile : found")
else:
    print("county shapefile : MISSING")
    print(f"  -> download Census 'cb_2023_us_county_500k' and unzip into {shp.parent}")
    print("     (only needed for notebook 02, not for collecting)")

## Step 4 — Check the registry

In [ ]:
import json
entries = json.loads(config.REGISTRY_PATH.read_text())
enabled = [e for e in entries if e.get("enabled")]
print(f"registry.json: {len(entries)} entries, {len(enabled)} enabled")
for e in enabled:
    print(f"  - {e['utility_id']:<20} ({e.get('platform','?')})")
if not enabled:
    print("  no utilities enabled yet - that is expected on a fresh clone.")
    print("  go to 01_build_registry.ipynb to add real utilities.")

---

If Steps 1-3 pass, the environment is ready. Next:

- **`01_build_registry.ipynb`** - discover, test, and register utilities.
- **`02_aggregate_and_verify.ipynb`** - aggregate to county and cross-check
  against EAGLE-I.